In [ ]:
import random
import matplotlib.pyplot as plt
import sltn
from sltn import Signature, Interpretation
from sltn import Logic, kb_describe
import sltn.interpretation.operators as fuzzy_ops
import torchvision
from utils import *

# Classifying image sequences wrt LTL$_f$ formulas

This test is inspired from the experiment described in Andreoni, Riccardo, et al. "T-ILR: a Neurosymbolic Integration for LTLf." Conference on Neurosymbolic Learning and Reasoning. PMLR, 2025.

We consider the following set of formulas introduced in Umili, Elena, et al. "Grounding LTL$_f$ specifications in image sequences." Proceedings of the International Conference on Principles of Knowledge Representation and Reasoning. Vol. 19. No. 1. 2023.


In [ ]:
ltlf_strings = [
"a",
"F a -> F b",
"G( a -> F b)",
"(! b U a) | G (! b)",
'G( a -> F b) & ((! b U a) | G (! b))',
'G( a -> X (! a U b))',
'(! b U a) | G(! b) & G (b -> X ((!b U a) | G (!b)))',
'G( a -> X (! a U b)) & (! b U a) | G (! b)',
'G(a -> X b)',
'G((X b) -> a)',
'!(F a & F b)',
'G (a -> ! (F b))',
'G(a -> ! (X b))',
'F a | F b',
"F(a & X ( F(a)))",
"!(F(a & X ( F(a))))",
"(F(a & X ( F(a)))) & !(  F(a & X ( F ( F (a & X( F(a)))))  )  )",
"(F(b)) -> (F(a))",
"G((a <-> X(b)))",
"(F a | F b) & ! (F a & F b)"
]

# Temporarily rewriting implication, until problems with implications are resolved

# ltlf_strings = [
# "a",
# "! F a | F b",
# "G( ! a | F b)",
# "(! b U a) | G (! b)",
# 'G( ! a | F b) & ((! b U a) | G (! b))',
# 'G( ! a | X (! a U b))',
# '(! b U a) | G(! b) & G (! b | X ((!b U a) | G (!b)))',
# 'G( !a | X (! a U b)) & (! b U a) | G (! b)',
# 'G(!a | X b)',
# 'G(!(X b) | a)',
# '!(F a & F b)',
# 'G (! a | ! (F b))',
# 'G(! a | ! (X b))',
# 'F a | F b',
# "F(a & X ( F(a)))",
# "!(F(a & X ( F(a))))",
# "(F(a & X ( F(a)))) & !(  F(a & X ( F ( F (a & X( F(a)))))  )  )",
# "! (F(b)) | (F(a))",
# "G((a <-> X(b)))",
# "(F a | F b) & ! (F a & F b)"
# ]

In [ ]:
nr_of_formulas_train = 1
nr_of_formulas_test = 1
ltlf_string = random.shuffle(ltlf_strings)
ltlf_strings_train = ltlf_strings[1:5]
# ltlf_strings_train = ltlf_strings[8:9]
ltlf_strings_test = ltlf_strings[5:9]
# ltlf_strings_test = ltlf_strings[8:9]
trace_length = 5
nr_of_train_samples = 1000
nr_of_test_samples = 500
classes = [3,7]
STABLE = True
EPS = 1e-12
P = 2
P_struct = 4
THRESHOLD = 0.5
batch_size = 32
nr_of_epochs = 20
learning_rate = 1e-4

device = torch.device("cuda:3" if torch.cuda.is_available() else "mps")
device

In [ ]:
logic = Logic()
logic["∧"] = fuzzy_ops.AndMin()
logic["¬"] = fuzzy_ops.NotStandard()
logic["∃"] = fuzzy_ops.AggregPMean(stable=STABLE, p=P, eps=EPS)
logic["∀"] = fuzzy_ops.AggregPMeanError(stable=STABLE, p=P, eps=EPS)
logic["SatAgg"] = fuzzy_ops.AggregPMean(stable=STABLE, p=P, eps=EPS)
logic["∀,t"] = fuzzy_ops.AggregMin()
logic["∃,t"] = fuzzy_ops.AggregMax()

$a$ and $b$ are considered as parameters that can be instantiated to a pair of digits. Following the protocol of the paper cited above for every set of $n$ symbols $\{d_1,\dots,d_n\}$ and for every formula $\phi(a,b)$ in the above set, we consider the formula
$$
\phi(d_1,d_2) \wedge \phi(d_2,d_3) \wedge\dots\wedge\phi(d_{n-1},d_n)
$$

In [ ]:
ltlf_atoms = {f"d{i}":i for i in classes}

grounded_ltlf_strings_train = []
for f in ltlf_strings_train:
    grounded_ltlf_strings_train.append(ground_formula_on_atoms(f, list(ltlf_atoms.keys())))

grounded_ltlf_strings_test = []
for f in ltlf_strings_test:
    grounded_ltlf_strings_test.append(ground_formula_on_atoms(f, list(ltlf_atoms.keys())))

In [ ]:
mnist_train = torchvision.datasets.MNIST(
    "../datasets/", train=True, download=True,
    transform=torchvision.transforms.ToTensor())

mnist_test = torchvision.datasets.MNIST(
    "../datasets/", train=False, download=True,
    transform=torchvision.transforms.ToTensor())

train_traces, train_labels = generate_data_sequence_dataset_for_ltlf(grounded_ltlf_strings_train, trace_length, nr_of_train_samples, mnist_train, ltlf_atoms)
test_traces, test_labels = generate_data_sequence_dataset_for_ltlf(grounded_ltlf_strings_test, trace_length, nr_of_test_samples, mnist_test,ltlf_atoms)

print("train data and labels shapes:", train_traces.shape,train_labels.shape)
print("Test data and labels shapes:", test_traces.shape,test_labels.shape)

In [ ]:
class DataLoader:
    def __init__(self, data, labels, batch_size=64, shuffle=True):
        self.data = data.to(device)
        self.labels = labels
        self.batch_size = batch_size
        self.shuffle = shuffle

    def __len__(self):
        return int(np.ceil(self.data.shape[0] / self.batch_size))

    def __iter__(self):
        idxlist = list(range(self.data.shape[0]))
        if self.shuffle:
            np.random.shuffle(idxlist)
        for start in range(0, self.data.shape[0], self.batch_size):
            idx = idxlist[start:start + self.batch_size]
            yield self.data[idx], self.labels[idx]

# instantiate the dataloaders
train_dataloader = DataLoader(train_traces, train_labels, batch_size=batch_size)
test_dataloader = DataLoader(test_traces, test_labels, batch_size=batch_size, shuffle=False)

def plot_trace(trace, cmap="gray"):
    T = len(trace)
    fig, axes = plt.subplots(1, T)
    for t, ax in enumerate(axes):
        ax.imshow(trace[t][0], cmap=cmap)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

for idx_f,ltlf_f in enumerate(grounded_ltlf_strings_train):
    for v in range(2):
        print(ltlf_f)
        idx_sample = idx_f*2*nr_of_train_samples + v*nr_of_train_samples
        for i in range(3):
            plot_trace(train_traces[idx_sample+10+i])
            print(train_labels[idx_sample+i])

# Representation of $LTL_f$ in structured FOL
In general modal logic can be translated in first order logic. In particular LTL$_f$ modal operators' semantics can be expressed in first order logic where every atom $a$ of LTL$_f$ formlua is replaced with a unary predicate $a(t)$, where $t$ is an individual variable that denotes time points of a linear structure. In this way we can translate temporal operators as follows:
| LTL$_f$ | FOL 2 |
|---|---|
| $X \phi$ | $\exists t':next(t,t') \wedge \phi(t')$ |
| $G \phi$ | $\forall t':after(t,t') \rightarrow \phi(t')$ |
| $F \phi$ | $\exists t':after(t,t') \wedge \phi(t')$ |
| $\phi U \psi$ | $\exists t': (\psi(t') \wedge \forall t'': (after(t,t'') \wedge  after(t'',t') \rightarrow \phi(t'')))$

To this aim we need to declare the non logical symbols in an sLTN signatore

In [ ]:
sig = Signature("ltlf")
sig.sort("Image")
sig.sort("multi_label")
sig.dimension("T")
sig.structural_variable("t", "T")
sig.variable("x", "Image", dims=["T"])
sig.variable("y", "multi_label")


for i in range(len(grounded_ltlf_strings_train)):
    sig.predicate(f"true_{i}",["multi_label"])

for a,at in enumerate(ltlf_atoms):
    sig.predicate(at,["Image"])

sig.structural_relation("After", ["T", "T"])
sig.structural_relation("Next", ["T", "T"])

We can now declare formulas with this signature:

In [ ]:
clauses = []
for i,f in enumerate(grounded_ltlf_strings_train):
    clauses.append(sig.parse(f"forall (x, y) : (true_{i}(y) <-> {ltl_to_fol(f,'x','t')})"))
kb = sltn.KB(clauses)

In [ ]:
interp = Interpretation(sig).to(device=device)
interp["Image"] = sltn.Type("mnist_image",shape=(1,28,28))
interp["Next"] = torch.diag(torch.ones(trace_length-1), diagonal=1)
interp["After"] = torch.triu(torch.ones((trace_length,trace_length)), diagonal=0)

cnn = MNISTCNN(nr_of_classes=len(ltlf_atoms))
for a,at in enumerate(ltlf_atoms):
    interp[at] = torch.nn.Sequential(cnn,Project(a)).to(device=device)
for i in range(len(grounded_ltlf_strings_train)):
    interp[f"true_{i}"] = Project(i).to(device=device)
interp.logic = logic

In [ ]:
print("Structured predicates:")
print("Next:")
# side by side plots
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].set_title("Next")
axes[1].set_title("After")

axes[0].spy(interp["Next"].cpu(), markersize=10, label="Next")
axes[0].set_xticks(range(trace_length))
axes[0].set_yticks(range(trace_length))
axes[0].grid()

axes[1].spy(interp["After"].cpu(), markersize=10, label="After")
axes[1].set_xticks(range(trace_length))
axes[1].set_yticks(range(trace_length))
axes[1].grid()
plt.show()

## Training the models of the predicates `d0`, `d1` and `d2`

In [ ]:
from tqdm.notebook import tqdm
from torchmetrics import MetricCollection
from torchmetrics.classification import BinaryPrecision, BinaryRecall,BinaryF1Score,BinaryAccuracy

def compute_metrics(ltlf_strings,interp,dataloader,sltn_strings=None):
    interp["multi_label"] = sltn.Type("multi_label",shape=(len(ltlf_strings),))
    if sltn_strings is None:
        sltn_strings = [ltl_to_fol(ltlf_str, 'x', 't') for ltlf_str in ltlf_strings]
    metrics = {f: MetricCollection({
        "accuracy": BinaryAccuracy().to(device=device),
        "precision": BinaryPrecision().to(device=device),
        "recall": BinaryRecall().to(device=device),
        "f1_score": BinaryF1Score().to(device=device),})
     for f in ltlf_strings}

    for traces, labels in tqdm(dataloader):
        interp["x"] = traces.to(device=device)
        labels = labels.to(device=device)
        for i,sltn_str in enumerate(sltn_strings):
            preds = (interp(sltn_str).value > THRESHOLD).to(torch.int).to(device=device).squeeze()
            metrics[ltlf_strings[i]](preds, labels[:,i])
    return {s:metrics[s].compute() for s in ltlf_strings}


In [ ]:
optimizer = torch.optim.Adam(cnn.parameters(), lr=learning_rate)
for epoch in range(20):
    interp["multi_label"] = sltn.Type("multi_label",shape=(len(grounded_ltlf_strings_train),))
    train_loss = 0.0
    cnn.train()
    for traces, labels in tqdm(train_dataloader):
        interp["x"] = traces.to(device=device)
        interp["y"] = labels.to(device=device)
        optimizer.zero_grad()
        loss = 1 - interp(kb).value
        loss.backward()
        # stats = kb_backward(optimizer=optimizer,interp=interp,formulas=kb_dict)
        # train_loss += stats["agg_loss"]
        optimizer.step()
        # train_loss += stats["agg_loss"]
        train_loss += loss.item()
    train_loss /= len(train_dataloader)
    print("loss: ", train_loss)
    if epoch % 10 == 0 or epoch == nr_of_epochs - 1:
        train_metrics = compute_metrics(grounded_ltlf_strings_train,interp,train_dataloader)
        test_metrics = compute_metrics(grounded_ltlf_strings_test,interp,test_dataloader)
        print("------train formulas evaluation on test traces-----")
        print("formula,acc,f1,prec,rec")
        for k,v in train_metrics.items():
            print(k,[str(v[k1].data.cpu().numpy()) for k1 in v])
        print("------test formulas evaluation on test traces-----")
        for k,v in test_metrics.items():
            print(k,[str(v[k1].data.cpu().numpy()) for k1 in v])